# Model 1: Simple 3D CNN

Baseline model for classifying cognitive status (cognitively normal [CN], mild cognitive impairment [MCI], or Alzheimer's disease [AD]) from a single 3D T1-weighted brain MRI volume. One model is trained for each target visit: baseline, 12 months, and 24 months.

> **Data notice:** The data used for this project are restricted and are not included in this repository. This notebook has been cleared of all references to the data and of all outputs. See `DATA_NOTICE.md`, and `README.md` for the expected input format.

### Load Data

Point `IMAGE_DIR` and `LABELS_PATH` to your own data. Each image file is named `<subject_id>.nii.gz`, and the labels table has one row per subject and visit.

In [ ]:
from pathlib import Path
from datetime import datetime

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, classification_report
from scipy.ndimage import zoom

from tqdm.auto import tqdm


In [ ]:
# Paths to your own data. The data used for this project are restricted and are not
# included in this repository (see DATA_NOTICE.md).
DATA_ROOT = Path("./data")
IMAGE_DIR = DATA_ROOT / "images"        # one preprocessed 3D NIfTI volume per subject: <subject_id>.nii.gz
LABELS_PATH = DATA_ROOT / "labels.csv"  # one row per subject and visit

# Column names in the labels table.
SUBJECT_COL = "subject_id"
VISIT_COL = "visit"
DIAGNOSIS_COL = "diagnosis"

labels_table = pd.read_csv(LABELS_PATH, dtype={SUBJECT_COL: str}, low_memory=False)

VALID_TARGET_VISITS = ("baseline", "month12", "month24")
SELECTED_TARGET_VISIT = "baseline"

label_map = {
    "CN": 0,
    "MCI": 1,
    "Dementia": 2,
    "AD": 2,
}


def build_metadata_for_target(target_visit):
    """Build image metadata and class labels for one outcome visit."""
    if target_visit not in VALID_TARGET_VISITS:
        raise ValueError(f"target_visit must be one of {VALID_TARGET_VISITS}, got {target_visit!r}")

    label_df = labels_table[labels_table[VISIT_COL] == target_visit][[SUBJECT_COL, DIAGNOSIS_COL]].copy()
    label_df = label_df.dropna(subset=[DIAGNOSIS_COL])
    label_df["label"] = label_df[DIAGNOSIS_COL].map(label_map)
    label_df = label_df.dropna(subset=["label"])
    label_df["label"] = label_df["label"].astype(int)

    rows = []
    for f in sorted(IMAGE_DIR.glob("*.nii.gz")):
        subject_id = f.name[: -len(".nii.gz")]
        rows.append({SUBJECT_COL: subject_id, "image_path": f})

    image_df = pd.DataFrame(rows, columns=[SUBJECT_COL, "image_path"])
    metadata = image_df.merge(label_df[[SUBJECT_COL, "label", DIAGNOSIS_COL]], on=SUBJECT_COL, how="inner")
    metadata["target_visit"] = target_visit
    return metadata


metadata = build_metadata_for_target(SELECTED_TARGET_VISIT)
metadata.head(), metadata[DIAGNOSIS_COL].value_counts()

In [ ]:
class NiftiMRIDataset(Dataset):
    def __init__(self, metadata, target_shape=(64, 64, 64)):
        self.metadata = metadata.reset_index(drop=True)
        self.target_shape = target_shape

    def __len__(self):
        return len(self.metadata)

    def _resize(self, arr):
        factors = [
            self.target_shape[0] / arr.shape[0],
            self.target_shape[1] / arr.shape[1],
            self.target_shape[2] / arr.shape[2],
        ]
        return zoom(arr, factors, order=1)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]

        img = nib.load(row["image_path"])
        arr = img.get_fdata(dtype=np.float32)

        arr = np.nan_to_num(arr)

        # Resize so every scan has same shape.
        arr = self._resize(arr)

        # Normalize each scan.
        mean = arr.mean()
        std = arr.std()
        arr = (arr - mean) / (std + 1e-6)

        # PyTorch 3D CNN format: [channel, depth, height, width]
        x = torch.tensor(arr, dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(row["label"], dtype=torch.long)

        return x, y
    
def make_splits(metadata, random_state=42):
    train_df, temp_df = train_test_split(
        metadata,
        test_size=0.30,
        random_state=random_state,
        stratify=metadata["label"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=1/3,   # 1/3 of 30% = 10%
        random_state=random_state,
        stratify=temp_df["label"]
    )

    return train_df, val_df, test_df


train_df, val_df, test_df = make_splits(metadata)

print(len(train_df), len(val_df), len(test_df))
print(train_df["label"].value_counts())
print(val_df["label"].value_counts())
print(test_df["label"].value_counts())


In [ ]:
batch_size = 4


def make_dataloaders(train_df, val_df, test_df, batch_size=4, target_shape=(64, 64, 64), num_workers=2):
    train_dataset = NiftiMRIDataset(train_df, target_shape=target_shape)
    val_dataset = NiftiMRIDataset(val_df, target_shape=target_shape)
    test_dataset = NiftiMRIDataset(test_df, target_shape=target_shape)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_dataloaders(
    train_df,
    val_df,
    test_df,
    batch_size=batch_size
)

x_batch, y_batch = next(iter(train_loader))

print(x_batch.shape)
print(y_batch.shape)
print(y_batch)


# CNN

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device
class Small3DCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, padding=1),
            nn.BatchNorm3d(8),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(8, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool3d(1)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

### Initialize model

In [ ]:
def initialize_training(model_cls=Small3DCNN, num_classes=3, lr=1e-4, weight_decay=1e-4):
    model = model_cls(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )
    return model, criterion, optimizer, scheduler


model, criterion, optimizer, scheduler = initialize_training()


In [ ]:
device

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    # progress_bar = tqdm(loader, desc="Training", leave=False)

    # for x, y in progress_bar:
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(y.detach().cpu().numpy())

        current_loss = total_loss / len(all_labels)
        # progress_bar.set_postfix(loss=f"{current_loss:.4f}")

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, acc, bal_acc, macro_f1

def evaluate(model, loader, criterion, device, desc="Evaluating"):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    # progress_bar = tqdm(loader, desc=desc, leave=False)

    with torch.no_grad():
        # for x, y in progress_bar:
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)

            total_loss += loss.item() * x.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            current_loss = total_loss / len(all_labels)
            # progress_bar.set_postfix(loss=f"{current_loss:.4f}")

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "macro_f1": macro_f1,
        "labels": np.array(all_labels),
        "preds": np.array(all_preds),
        "probs": np.array(all_probs),
    }


### Full training loop

In [ ]:
num_epochs = 30
training_started_at = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint_root = Path("models") / training_started_at
checkpoint_root.mkdir(parents=True, exist_ok=True)

training_targets = list(VALID_TARGET_VISITS)
training_results = {}
all_histories = []

for target_visit in training_targets:
    print(f"\nTraining target_visit={target_visit}")

    target_metadata = build_metadata_for_target(target_visit)
    target_train_df, target_val_df, target_test_df = make_splits(target_metadata)
    target_train_loader, target_val_loader, target_test_loader = make_dataloaders(
        target_train_df,
        target_val_df,
        target_test_df,
        batch_size=batch_size
    )

    model, criterion, optimizer, scheduler = initialize_training()
    best_val_f1 = -1
    target_checkpoint_dir = checkpoint_root / target_visit
    target_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = target_checkpoint_dir / "best_3d_cnn.pt"
    history = []

    for epoch in range(num_epochs):
        train_loss, train_acc, train_bal_acc, train_f1 = train_one_epoch(
            model,
            target_train_loader,
            optimizer,
            criterion,
            device
        )

        val_results = evaluate(
            model,
            target_val_loader,
            criterion,
            device
        )

        scheduler.step(val_results["macro_f1"])

        row = {
            "target_visit": target_visit,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "train_balanced_accuracy": train_bal_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_results["loss"],
            "val_accuracy": val_results["accuracy"],
            "val_balanced_accuracy": val_results["balanced_accuracy"],
            "val_macro_f1": val_results["macro_f1"],
        }

        history.append(row)
        all_histories.append(row)

        print(
            f"{target_visit} | "
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_results['loss']:.4f} | "
            f"Val F1: {val_results['macro_f1']:.4f}"
        )

        if val_results["macro_f1"] > best_val_f1:
            best_val_f1 = val_results["macro_f1"]
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved best model to {best_model_path}")

    training_results[target_visit] = {
        "metadata": target_metadata,
        "train_df": target_train_df,
        "val_df": target_val_df,
        "test_df": target_test_df,
        "train_loader": target_train_loader,
        "val_loader": target_val_loader,
        "test_loader": target_test_loader,
        "history": history,
        "best_val_f1": best_val_f1,
        "best_model_path": best_model_path,
    }

print(f"\nAll best models saved under: {checkpoint_root}")


In [ ]:
history_df = pd.DataFrame(all_histories)
history_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for target_visit, target_history in history_df.groupby("target_visit"):
    ax.plot(
        target_history["epoch"].to_numpy(dtype=float),
        target_history["train_macro_f1"].to_numpy(dtype=float),
        marker="o",
        linewidth=2,
        label=f"{target_visit} Train Macro F1"
    )
    ax.plot(
        target_history["epoch"].to_numpy(dtype=float),
        target_history["val_macro_f1"].to_numpy(dtype=float),
        marker="o",
        linewidth=2,
        linestyle="--",
        label=f"{target_visit} Validation Macro F1"
    )

ax.set_xlabel("Epoch")
ax.set_ylabel("Macro F1")
ax.set_title("Training Curves")
ax.set_ylim(0, 1)
ax.set_xticks(range(1, num_epochs + 1))
ax.grid(True, alpha=0.3)
ax.legend()

plt.show()


In [ ]:
test_results_by_target = {}

for target_visit, result in training_results.items():
    best_model = Small3DCNN(num_classes=3).to(device)
    best_model.load_state_dict(torch.load(result["best_model_path"], map_location=device))

    test_results = evaluate(
        best_model,
        result["test_loader"],
        criterion,
        device
    )

    test_results_by_target[target_visit] = test_results

    print(f"\nTarget visit: {target_visit}")
    print("Best model path:", result["best_model_path"])
    print("Test Accuracy:", test_results["accuracy"])
    print("Test Balanced Accuracy:", test_results["balanced_accuracy"])
    print("Test Macro F1:", test_results["macro_f1"])

best_target_visit = max(
    training_results,
    key=lambda target_visit: training_results[target_visit]["best_val_f1"]
)
best_model_path = training_results[best_target_visit]["best_model_path"]
test_results = test_results_by_target[best_target_visit]

print(f"\nBest target by validation Macro F1: {best_target_visit}")


In [ ]:
class_names = ["CN", "MCI", "AD"]

for target_visit, target_test_results in test_results_by_target.items():
    cm = confusion_matrix(
        target_test_results["labels"],
        target_test_results["preds"]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"True {c}" for c in class_names],
        columns=[f"Pred {c}" for c in class_names]
    )

    print(f"\nTarget visit: {target_visit}")
    print(cm_df)
    print(classification_report(
        target_test_results["labels"],
        target_test_results["preds"],
        target_names=class_names
    ))
